# Pattern ③ MCP Server連携
## 概要
DifyはHTTPベースのMCPサービスに対応しています。
DatabricksのManaged MCPサーバーを通じて、UC関数やVector Searchをエージェントのツールとして使えます。
## アーキテクチャ
```
Dify Agent
  │
  ├── MCP Protocol ──▶ Databricks Managed MCP Server
  │                       ├── UC Functions (ツール)
  │                       ├── Vector Search (検索)
  │                       └── Genie Spaces (データ分析)
  │
  └── UCガバナンスが維持されたまま外部公開
```

## 前提条件

- `00_data_setup.ipynb` 実行済み（テーブル・knowledge_base）
- `02_http_api_integration.ipynb` 実行済み（UC関数・Vector Search Index）

In [ ]:
%run ./_config

## Step 1: Databricks Managed MCP URL

In [ ]:
# MCP Server URLs
mcp_functions_url = f"https://{host}/api/2.0/mcp/functions/{catalog}/{schema}"
mcp_vs_url = f"https://{host}/api/2.0/mcp/vector-search/{catalog}/{schema}"

print("=" * 60)
print("Databricks Managed MCP Server URLs")
print("=" * 60)
print(f"\n【UC Functions MCP】")
print(f"  {mcp_functions_url}")
print(f"\n【Vector Search MCP】")
print(f"  {mcp_vs_url}")
print(f"\n【認証】Bearer {token[:10]}...")

## Step 2: MCPサーバーのツール一覧を確認

MCP Streamable HTTP プロトコルは JSON-RPC 形式で通信します。  
`databricks_mcp` パッケージを使わず、REST APIで直接呼び出します。

In [ ]:
import requests
import json

headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json",
    "Accept": "application/json, text/event-stream"
}

# MCP初期化: initialize
init_resp = requests.post(mcp_functions_url, headers=headers, json={
    "jsonrpc": "2.0",
    "method": "initialize",
    "params": {
        "protocolVersion": "2025-03-26",
        "capabilities": {},
        "clientInfo": {"name": "notebook-test", "version": "1.0"}
    },
    "id": 1
})
print("Initialize:", init_resp.status_code)
init_data = init_resp.json()
print(json.dumps(init_data, indent=2, ensure_ascii=False)[:500])

In [ ]:
# MCP tools/list: UC Functions のツール一覧を取得
tools_resp = requests.post(mcp_functions_url, headers=headers, json={
    "jsonrpc": "2.0",
    "method": "tools/list",
    "id": 2
})

tools_data = tools_resp.json()
tools = tools_data.get("result", {}).get("tools", [])
print(f"✅ UC Functions MCP — {len(tools)} ツール:")
for t in tools:
    desc = (t.get("description") or "")[:60]
    print(f"  - {t['name']}: {desc}")

In [ ]:
# MCP tools/call: search_policy を呼び出し
call_resp = requests.post(mcp_functions_url, headers=headers, json={
    "jsonrpc": "2.0",
    "method": "tools/call",
    "params": {
        "name": f"{catalog}__{schema}__search_policy",
        "arguments": {"keyword": "返品"}
    },
    "id": 3
})

call_data = call_resp.json()
result = call_data.get("result", {})
print("✅ MCP経由の検索結果:")
for content in result.get("content", []):
    print(content.get("text", "")[:300])

## Step 3: Python UC関数のテスト（calculate_math_expression）

In [ ]:
# Python UC関数: calculate_math_expression
math_resp = requests.post(mcp_functions_url, headers=headers, json={
    "jsonrpc": "2.0",
    "method": "tools/call",
    "params": {
        "name": f"{catalog}__{schema}__calculate_math_expression",
        "arguments": {"expression": "sqrt(2 + 3 * 4)"}
    },
    "id": 4
})

math_data = math_resp.json()
math_result = math_data.get("result", {})
print("✅ MCP経由のPython UC関数:")
for content in math_result.get("content", []):
    print(f"  sqrt(2 + 3 * 4) = {content.get('text', '')}")

## Step 4: Vector Search MCP の接続テスト

In [ ]:
# Vector Search MCP: initialize + tools/list
vs_init = requests.post(mcp_vs_url, headers=headers, json={
    "jsonrpc": "2.0",
    "method": "initialize",
    "params": {
        "protocolVersion": "2025-03-26",
        "capabilities": {},
        "clientInfo": {"name": "notebook-test", "version": "1.0"}
    },
    "id": 1
})
print(f"VS MCP Initialize: {vs_init.status_code}")

vs_tools_resp = requests.post(mcp_vs_url, headers=headers, json={
    "jsonrpc": "2.0",
    "method": "tools/list",
    "id": 2
})

vs_tools_data = vs_tools_resp.json()
vs_tools = vs_tools_data.get("result", {}).get("tools", [])
print(f"\n✅ Vector Search MCP — {len(vs_tools)} ツール:")
for t in vs_tools:
    print(f"  - {t['name']}: {(t.get('description') or '')[:80]}")

In [ ]:
# Vector Search MCP: tools/call
vs_index_fullname = f"{catalog}.{schema}.knowledge_base_vs_index"

# ツール名はlist結果から取得
vs_tool_name = vs_tools[0]["name"] if vs_tools else "vector_search_query"
print(f"ツール名: {vs_tool_name}")

# MCP経由のVector Searchは引数が異なる（query_text ではなく query）
# _meta で num_results, columns を指定
vs_call = requests.post(mcp_vs_url, headers=headers, json={
    "jsonrpc": "2.0",
    "method": "tools/call",
    "params": {
        "name": vs_tool_name,
        "arguments": {
            "query": "SmartWatch X100のバッテリー",
            "_meta": {
                "num_results": "2",
                "columns": "content,doc_uri"
            }
        }
    },
    "id": 3
})

vs_call_data = vs_call.json()
if "error" in vs_call_data:
    print(f"❌ エラー: {json.dumps(vs_call_data['error'], ensure_ascii=False)[:300]}")
else:
    vs_result = vs_call_data.get("result", {})
    print("✅ MCP経由のVector Search結果:")
    for content in vs_result.get("content", []):
        print(content.get("text", "")[:400])

## Genie MCP

Databricks Managed MCPはGenie Spacesもサポートしています。Genie SpaceをMCPサーバーとして公開し、自然言語でSQLデータにアクセスできます。

MCP URL: `https://{host}/api/2.0/mcp/genie/{GENIE_SPACE_ID}`

In [ ]:
# Genie MCP の接続テスト
if GENIE_SPACE_ID:
    genie_mcp_url = f"https://{host}/api/2.0/mcp/genie/{GENIE_SPACE_ID}"
    print(f"Genie MCP URL: {genie_mcp_url}")

    # Initialize
    genie_init = requests.post(genie_mcp_url, headers=headers, json={
        "jsonrpc": "2.0",
        "method": "initialize",
        "params": {
            "protocolVersion": "2025-03-26",
            "capabilities": {},
            "clientInfo": {"name": "notebook-test", "version": "1.0"}
        },
        "id": 1
    })
    print(f"Initialize: {genie_init.status_code}")

    # tools/list
    genie_tools_resp = requests.post(genie_mcp_url, headers=headers, json={
        "jsonrpc": "2.0",
        "method": "tools/list",
        "id": 2
    })
    genie_tools = genie_tools_resp.json().get("result", {}).get("tools", [])
    print(f"\n✅ Genie MCP — {len(genie_tools)} ツール:")
    for t in genie_tools:
        print(f"  - {t['name']}: {(t.get('description') or '')[:80]}")
else:
    print("⚠️ GENIE_SPACE_ID が未設定です。config.yaml で設定してください。")

In [ ]:
import time

# Genie MCP: tools/call + ポーリング
if GENIE_SPACE_ID and genie_tools:
    genie_tool_name = genie_tools[0]["name"]
    print(f"ツール名: {genie_tool_name}")

    # Step 1: クエリ送信
    genie_call = requests.post(genie_mcp_url, headers=headers, json={
        "jsonrpc": "2.0",
        "method": "tools/call",
        "params": {
            "name": genie_tool_name,
            "arguments": {"query": "商品は何件ありますか？"}
        },
        "id": 3
    })
    genie_call_data = genie_call.json()
    result_text = genie_call_data.get("result", {}).get("content", [{}])[0].get("text", "")
    result_json = json.loads(result_text) if result_text.startswith("{") else {}

    conv_id = result_json.get("conversationId", "")
    msg_id = result_json.get("messageId", "")
    status = result_json.get("status", "")
    print(f"初回応答: status={status}, conv={conv_id}, msg={msg_id}")

    # Step 2: ポーリング（poll_response ツールを使用）
    poll_tool_name = None
    for t in genie_tools:
        if t["name"].startswith("poll_response"):
            poll_tool_name = t["name"]
            break

    if poll_tool_name and conv_id and msg_id:
        print(f"ポーリングツール: {poll_tool_name}")
        for attempt in range(15):
            time.sleep(3)
            poll_resp = requests.post(genie_mcp_url, headers=headers, json={
                "jsonrpc": "2.0",
                "method": "tools/call",
                "params": {
                    "name": poll_tool_name,
                    "arguments": {
                        "conversation_id": conv_id,
                        "message_id": msg_id
                    }
                },
                "id": 4 + attempt
            })
            poll_data = poll_resp.json()
            poll_text = poll_data.get("result", {}).get("content", [{}])[0].get("text", "")
            poll_json = json.loads(poll_text) if poll_text.startswith("{") else {}
            poll_status = poll_json.get("status", "")
            print(f"  Polling {attempt+1}: {poll_status}")

            if poll_status in ["COMPLETED", "FAILED", "CANCELLED"]:
                if poll_status == "COMPLETED":
                    text_attachments = poll_json.get("content", {}).get("textAttachments", [])
                    query_attachments = poll_json.get("content", {}).get("queryAttachments", [])
                    if text_attachments:
                        print(f"\n✅ Genie回答: {text_attachments[0][:300]}")
                    if query_attachments:
                        print(f"\n  生成SQL: {str(query_attachments[0])[:200]}")
                else:
                    print(f"❌ {poll_status}: {poll_text[:200]}")
                break
    else:
        print("⚠️ ポーリングツールが見つかりません")

## Step 5: Dify側の設定

### 5.1 MCPプラグインのインストール
1. Difyの **Plugins** → **Marketplace** を開く
2. **「MCP SSE」** (`junjiem/mcp_sse`) を検索してインストール
   - Streamable HTTP トランスポートに対応

### 5.2 MCPサーバーの認証設定
1. **Tools** → **MCP SSE / StreamableHTTP** → **Authorize** をクリック
2. **認証名**: `Databricks Managed MCP` と入力
3. **MCP Servers config**: デフォルト内容を全て消して、下のセルで出力されるJSONを貼り付け
4. **保存** をクリック → UC Functions（5つ）とVector Search（1つ）のツールが自動発見される

In [ ]:
import json

# Difyに設定するMCPサーバー設定JSON（PATトークン埋め込み済み）
dify_mcp_config = {
    "databricks_uc_functions": {
        "transport": "streamable_http",
        "url": mcp_functions_url,
        "headers": {
            "Authorization": f"Bearer {token}"
        },
        "timeout": 50
    },
    "databricks_vector_search": {
        "transport": "streamable_http",
        "url": mcp_vs_url,
        "headers": {
            "Authorization": f"Bearer {token}"
        },
        "timeout": 50
    }
}

# Genie MCP を追加（GENIE_SPACE_ID が設定されている場合）
if GENIE_SPACE_ID:
    genie_mcp_url = f"https://{host}/api/2.0/mcp/genie/{GENIE_SPACE_ID}"
    dify_mcp_config["databricks_genie"] = {
        "transport": "streamable_http",
        "url": genie_mcp_url,
        "headers": {
            "Authorization": f"Bearer {token}"
        },
        "timeout": 120
    }

print("=" * 60)
print("Dify MCP SSE プラグイン設定 JSON（そのままコピー&ペースト可能）")
print("=" * 60)
print(json.dumps(dify_mcp_config, indent=2, ensure_ascii=False))
print(f"\n【MCPプロトコル】")
print(f"  JSON-RPC over Streamable HTTP")
print(f"\n【注意】")
print(f"  DifyのSSRF Proxyが .cloud.databricks.com を許可していること")
print(f"  PATトークンの有効期限に注意（期限切れ時は再設定が必要）")

### 5.3 エージェントアプリの作成
1. **Studio** → **Create from Blank** → **Agent**
2. **Tools** セクションで MCP SSE ツールを追加:
   - `mcp_sse_list_tools`（ツール一覧取得）
   - `mcp_sse_call_tool`（ツール実行）
3. System Prompt:
   ```
   あなたはTechNovaのカスタマーサポートです。
   利用可能なツールを使って、顧客の質問に正確に回答してください。
   回答は日本語で行ってください。

   まず mcp_sse_list_tools でツール一覧を取得し、適切なツールを mcp_sse_call_tool で呼び出してください。

   Genie（query_space_*）を使う場合：
   - query_space_* でクエリを送信すると、statusが FILTERING_CONTEXT 等で返ります
   - 返されたconversation_idとmessage_idを使って poll_response_* を呼び出してください
   - statusが COMPLETED になるまで poll_response_* を繰り返し呼んでください
   ```
4. **max_iteration** を `15` 以上に設定（Genieのポーリング回数分）
5. **Publish** → テスト

### テスト質問
- 「返品ポリシーについて教えてください」（→ UC Functions経由）
- 「SmartWatch X100のバッテリー持続時間は？」（→ Vector Search経由）
- 「商品は何件ありますか？」（→ Genie MCP経由）

## メリット
| 観点 | 説明 |
|------|------|
| 標準プロトコル | MCPプロトコルでツール連携が標準化（Streamable HTTP） |
| 一元管理 | ツール追加・変更はDatabricks側のみ |
| ガバナンス | UC権限がMCP経由でも維持 |
| UC Functions + Vector Search | 両方をMCPで外部公開可能 |
| 低コード | Dify側でURL設定のみ。コード変更不要 |

---
## DSLインポート（オプション）

上記の手動設定の代わりに、DSLファイルからエージェントを復元できます。

1. Dify → **Studio** → **Import DSL File** → `dsl/03_MCP_Agent.yml`

> MCP SSEプラグインのAuthorize設定は上記Step 5で実施済みのため、インポート後の追加設定は不要です。

![MCP Agent実行画面](../images/03_mcp_agent.jpg)
*Pattern ③ MCP Agent（UC Functions + Vector Search + Genie）*